# 3. The api of the webapp

The project's own server (`task serve` at home, or the demo space) answers
under `/api`, described at `/openapi.json`. The same api feeds the website,
the desktop window, the browser extension and the MCP server.

The dataset is public: nothing here needs a token. Run it with
`uv run --project examples --with jupyterlab jupyter lab examples/notebooks` (or the marimo version: `task examples:convert`, then
version: `task examples:convert`, then
`uv run --project examples marimo edit examples/marimo`).

`P_JOL_ROLL_API` points it elsewhere; by default it asks the demo space.

In [ ]:
import os

import pandas as pd
import requests

API = os.environ.get("P_JOL_ROLL_API", "https://sarverott-docs-of-law-by-gov-in-poland.hf.space").rstrip("/")


def get(path, **params):
    answer = requests.get(f"{API}{path}", params=params, timeout=120)
    answer.raise_for_status()
    return answer


try:
    journals = get("/api/journals").json()
    print([(journal["code"], len(journal["years"])) for journal in journals])
except Exception as error:  # the space may be asleep or starting
    journals = []
    print(f"{API} does not answer now:", error)

## Searching titles of a year

In [ ]:
if journals:
    found = get("/api/acts/DU/2025", q="podatku", limit=20).json()
    print(found["total"], "acts with this word in the title")
    shown = pd.DataFrame(found["items"], columns=["pos", "title", "status"])
else:
    shown = "no answer from the api"
shown

## One act and its ties

In [ ]:
if journals:
    view = get("/api/acts/DU/2016/1").json()
    print(view["act"]["title"])
    print("files:", view["files"], "- markdown:", view["markdown"])
    ties = pd.DataFrame(get("/api/acts/DU/2016/1/references").json())
else:
    ties = "no answer from the api"
ties

A raw file, `/api/acts/DU/2016/1/file/.pdf`, is served by the server at
home and redirected to the bucket by the demo.